# Week 4 participant practical: persistent homology as homology with memory

This is the student investigation, built around a small class that dies under inclusion, followed by a noisy-circle barcode. The setup and mathematical framing are supplied. You must record predictions, complete short computational steps, check intermediate objects and justify an interpretation.

We move through

$$K_a\subseteq K_b\longrightarrow H_p(K_a;\mathbb F_2)\to H_p(K_b;\mathbb F_2)
\longrightarrow\text{persistence module}\longrightarrow\text{barcode}.$$

The hand calculations come first. A library calculation is used only after the spaces, maps and interval summary have been identified.

See the course **Applied glossary** for translations of *induced map*, *birth*, *death*, *persistence module*, *barcode* and *essential class*.

**Working rule.** Run one section at a time. Before each TODO, state what shape, dimension or direction you expect in the output. Optional extensions come only after the core checkpoints agree.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RNG=np.random.default_rng(3024)

def rank_mod2(A):
    A=np.array(A,dtype=np.uint8,copy=True)%2; row=rank=0
    for col in range(A.shape[1]):
        piv=np.flatnonzero(A[row:,col])
        if not len(piv): continue
        q=row+piv[0]; A[[row,q]]=A[[q,row]]
        for i in range(A.shape[0]):
            if i!=row and A[i,col]: A[i]^=A[row]
        row+=1; rank+=1
        if row==A.shape[0]: break
    return rank

def plot_barcode(diagrams, titles=('H0','H1')):
    fig,axes=plt.subplots(1,len(diagrams),figsize=(10,3.4))
    for dim,(ax,D) in enumerate(zip(np.atleast_1d(axes),diagrams)):
        finite=D[np.isfinite(D[:,1])] if len(D) else D
        cap=max(finite[:,1].max() if len(finite) else 1, D[:,0].max() if len(D) else 1)*1.08
        for y,(b,d) in enumerate(D): ax.hlines(y,b,cap if np.isinf(d) else d,lw=2.5)
        ax.set_title(titles[dim]); ax.set_xlabel('Rips distance threshold ε'); ax.set_yticks([]); ax.set_xlim(left=0)
    plt.tight_layout(); plt.show()

print('Using coefficients in F_2.')

## 1. Observe: participant checkpoint

Let $X$ be the outline triangle and $Y$ the same vertices and edges plus the triangular face. The inclusion $X\hookrightarrow Y$ preserves every old simplex and chain.

The question is not whether the old edge cycle still exists as a chain. It does. The question is what happens to its class after $Y$ supplies a new 2-chain.

## 2. Predict: participant checkpoint

Before computing:

1. Predict $H_1(X;\mathbb F_2)$ and $H_1(Y;\mathbb F_2)$.
2. Is the induced map $H_1(X)\to H_1(Y)$ injective?
3. Two vertices merge after an edge is added. Which direction in $H_0\cong\mathbb F_2^2$ is killed?
4. If two unrelated classes live on $[1,4)$ and $[2,6)$, what is the dimension of the module at parameters 0, 1.5, 3 and 5?

## 3. Implement: participant checkpoint

### A. A class dies when it becomes a boundary

The cycle space of the three-edge outline is one-dimensional. Compare the image of $\partial_2$ before and after the face is added.

In [ ]:
d2_X=np.zeros((3,0),dtype=np.uint8)
d2_Y=np.ones((3,1),dtype=np.uint8)
dim_Z1=1
beta1_X=dim_Z1-rank_mod2(d2_X)
beta1_Y=dim_Z1-rank_mod2(d2_Y)
print('beta_1(X)=',beta1_X,'beta_1(Y)=',beta1_Y)

### B. Components merge through a non-injective map

In bases $(e_1,e_2)$ before the edge and $(e)$ afterwards, the induced map is represented by $[1\ 1]$.

In [ ]:
H0_map=np.array([[1,1]],dtype=np.uint8)
for v in [np.array([1,0]),np.array([0,1]),np.array([1,1])]:
    print(v,'maps to',(H0_map@v)%2)

## 4. Compare: participant checkpoint

### A. Betti counts versus module maps

Two modules can have the same dimension at every sampled parameter but connect their vector spaces differently. A sequence of Betti numbers records only vertical slice sizes. The persistence module retains the compatible maps.

For the two intervals $[1,4)$ and $[2,6)$, count how many bars cross each requested parameter.

In [ ]:
bars=[(1.,4.,'A'),(2.,6.,'B')]
fig,ax=plt.subplots(figsize=(8,2.8))
for y,(b,d,label) in enumerate(bars):
    ax.hlines(y,b,d,lw=5); ax.text(d+.12,y,label,va='center')
ax.set_xlim(0,7); ax.set_yticks([]); ax.set_xlabel('filtration parameter'); ax.set_title('Two interval summands'); plt.show()

def dimension_at(a): return sum(b<=a<d for b,d,_ in bars)
for a in [0,1.5,3,5]: print(a,dimension_at(a))

### Triangle filtration checkpoint

Consider the order $v_0,v_1,e_{01},v_2,e_{12},e_{02},f_{012}$.

| Addition | Predict the event |
|---|---|
| $v_0,v_1,v_2$ | |
| $e_{01},e_{12}$ | |
| $e_{02}$ | |
| $f_{012}$ | |

Name the two edge-component pairings and the edge-face pairing. Explain why the face kills a class rather than deleting its edge-cycle chain.

### B. From hand calculation to a library barcode

We now use `ripser` on two deterministic synthetic data sets: a noisy circle and a filled disk. `ripser` reports the Rips **pairwise-distance threshold** $\varepsilon$, so $\varepsilon=2r$ relative to Week 3's ball-radius convention.

Before running the cell, predict which data set should have the larger maximum finite $H_1$ persistence. Also inspect the radial coefficient of variation as a simpler baseline.

In [ ]:
n=90
theta=np.linspace(0,2*np.pi,n,endpoint=False)
circle=np.c_[np.cos(theta),np.sin(theta)]+0.035*RNG.normal(size=(n,2))
u=RNG.random(n); phi=2*np.pi*RNG.random(n)
disk=np.c_[np.sqrt(u)*np.cos(phi),np.sqrt(u)*np.sin(phi)]

fig,axes=plt.subplots(1,2,figsize=(7,3.2))
for ax,P,title in zip(axes,[circle,disk],['noisy circle','filled disk']):
    ax.scatter(P[:,0],P[:,1],s=14); ax.set_aspect('equal'); ax.set_title(title); ax.axis('off')
plt.show()

for name,P in [('circle',circle),('disk',disk)]:
    radii=np.linalg.norm(P-P.mean(axis=0),axis=1)
    print(name,'radial coefficient of variation =',round(radii.std()/radii.mean(),3))

In [ ]:
from ripser import ripser

# TODO: compute ripser(circle, maxdim=1)['dgms'] and the corresponding result for disk.
# TODO: plot both barcodes with plot_barcode.
# TODO: compare the longest finite H1 persistence in each result.

## 5. Interpret: participant checkpoint

1. Why can inclusion of complexes induce a non-injective map on homology?
2. What information do module maps retain that Betti numbers discard?
3. Under what one-parameter finiteness or tameness conditions is a barcode a complete interval summary?
4. What does an infinite death mean in this computed filtration, and when could it instead reflect truncation?
5. Did the topological calculation add anything beyond the radial baseline for this deliberately simple comparison?
6. Which Week 6 question begins once two diagrams need to be compared?

**◇ Object check.** The noisy samples, Rips filtration, persistence module, barcode and longest-bar scalar are different objects. Each arrow discards or adds information.

**† Qualification.** These are synthetic demonstrations, not empirical evidence about a dynamical system.